In [12]:
#Installing specific versions because previously it was giving issues. This keeps things close to what colab expects.

!pip install -q \
  pandas==2.2.2 \
  google-auth==2.47.0 \
  gcsfs==2025.3.0 \
  fsspec==2025.3.0 \
  google-cloud-storage==2.19.0

In [13]:
#Authenticating google account in colab

from google.colab import auth
auth.authenticate_user()

In [14]:
import google.auth
import gcsfs
import pandas as pd

#Grabbing credentials from colab

creds, project = google.auth.default()
print("Project:", project)

#Creating a file system storage for Google Cloud Storage

fs = gcsfs.GCSFileSystem(token=creds)

file_path = "gs://cs-562-aramark-project/Andrew_Meszaros_SRF_2026-04-01-0936.csv"

#Opening file from GCS and reading it in binary
#Streams the data from the cloud using file stream f

with fs.open(file_path, "rb") as f:
    df_sample = pd.read_csv(f, nrows=5)

df_sample

Project: 


,Year Name,Year Month,Business Entity Type,Country,Customer Market Segment Id,Client ID,Customer Id,Customer Brand Id,Customer Brand Parent Id,City,...,Ecommerce Status,Distributor ID,Distributor Group,Manufacturer ID,Category ID,Category Level 1,Category Level 2,Category Level 3,Category Level 4,Spend Random Factor
0,2025,202510,GPO,US,MS-100025,A0083918,C-CL0463-1,INDE,A0064145,PINEHURST,...,NaN,100114,TURF MANAGEMENT,M-105168,71.34.20.51.00,GOLF EQUIPMENT AND SUPPLIES,SPECIALIZED VEHICLES,SPECIALIZED VEHICLES,TURF UTILITY VEHICLES,1.237354e+06
1,2025,202512,GPO,US,MS-100026,A0091384,C-A0098350-1,WSVC,A0091384,LAHAINA,...,NaN,100079,ENGINEERING CLEANING SERVICES,M-100479,64.17.00.00.00,CHEMICALS AND CLEANING,CLEANING AND JANITORIAL SERVICES,NaN,NaN,6.002968e+05
2,2025,202507,MANAGED SERVICES,US,MS-100007,CF0000100,C-000010698-1,MS-100007,NaN,GUILFORD,...,ACTIVE,105067,ENGINEERING OTHER,NaN,65.41.00.00.00,MAINTENANCE AND ENGINEERING,M&E SERVICES,NaN,NaN,5.862804e+05
3,2025,202505,GPO,US,MS-100026,A0091384,C-A0098350-1,WSVC,A0091384,LAHAINA,...,NaN,100079,ENGINEERING CLEANING SERVICES,M-100479,64.17.00.00.00,CHEMICALS AND CLEANING,CLEANING AND JANITORIAL SERVICES,NaN,NaN,5.756939e+05
4,2025,202510,GPO,US,MS-100026,A0091384,C-A0098350-1,WSVC,A0091384,LAHAINA,...,NaN,100079,ENGINEERING CLEANING SERVICES,M-100479,64.17.00.00.00,CHEMICALS AND CLEANING,CLEANING AND JANITORIAL SERVICES,NaN,NaN,5.753405e+05


In [15]:
#Chunking data

chunk_iter = pd.read_csv(
    fs.open(file_path, "rb"),
    chunksize=100000
)

first_chunk = next(chunk_iter)
first_chunk.head()

,Year Name,Year Month,Business Entity Type,Country,Customer Market Segment Id,Client ID,Customer Id,Customer Brand Id,Customer Brand Parent Id,City,...,Ecommerce Status,Distributor ID,Distributor Group,Manufacturer ID,Category ID,Category Level 1,Category Level 2,Category Level 3,Category Level 4,Spend Random Factor
0,2025,202510,GPO,US,MS-100025,A0083918,C-CL0463-1,INDE,A0064145,PINEHURST,...,NaN,100114.0,TURF MANAGEMENT,M-105168,71.34.20.51.00,GOLF EQUIPMENT AND SUPPLIES,SPECIALIZED VEHICLES,SPECIALIZED VEHICLES,TURF UTILITY VEHICLES,1.237354e+06
1,2025,202512,GPO,US,MS-100026,A0091384,C-A0098350-1,WSVC,A0091384,LAHAINA,...,NaN,100079.0,ENGINEERING CLEANING SERVICES,M-100479,64.17.00.00.00,CHEMICALS AND CLEANING,CLEANING AND JANITORIAL SERVICES,NaN,NaN,6.002968e+05
2,2025,202507,MANAGED SERVICES,US,MS-100007,CF0000100,C-000010698-1,MS-100007,NaN,GUILFORD,...,ACTIVE,105067.0,ENGINEERING OTHER,NaN,65.41.00.00.00,MAINTENANCE AND ENGINEERING,M&E SERVICES,NaN,NaN,5.862804e+05
3,2025,202505,GPO,US,MS-100026,A0091384,C-A0098350-1,WSVC,A0091384,LAHAINA,...,NaN,100079.0,ENGINEERING CLEANING SERVICES,M-100479,64.17.00.00.00,CHEMICALS AND CLEANING,CLEANING AND JANITORIAL SERVICES,NaN,NaN,5.756939e+05
4,2025,202510,GPO,US,MS-100026,A0091384,C-A0098350-1,WSVC,A0091384,LAHAINA,...,NaN,100079.0,ENGINEERING CLEANING SERVICES,M-100479,64.17.00.00.00,CHEMICALS AND CLEANING,CLEANING AND JANITORIAL SERVICES,NaN,NaN,5.753405e+05


In [16]:
first_chunk.columns.tolist()

['Year Name',
 'Year Month',
 'Business Entity Type',
 'Country',
 'Customer Market Segment Id',
 'Client ID',
 'Customer Id',
 'Customer Brand Id',
 'Customer Brand Parent Id',
 'City',
 'State',
 'Zip',
 'Number of Rooms',
 'Ecommerce Status',
 'Distributor ID',
 'Distributor Group',
 'Manufacturer ID',
 'Category ID',
 'Category Level 1',
 'Category Level 2',
 'Category Level 3',
 'Category Level 4',
 'Spend Random Factor']

## Initial Data Exploration and Structure Understanding

At this stage, we successfully accessed the dataset stored in Google Cloud Storage and loaded a sample of the data using a chunk-based approach. Since the dataset contains approximately 44 million rows, it is not feasible to load the entire dataset into memory at once. Therefore, we used streaming via gcsfs and processed only a subset for initial exploration.

- The dataset contains 23 columns, representing a mix of:
  - Temporal features (e.g., Year, Year Month)
  - Organizational features (e.g., Business Entity Type, Client ID)
  - Geographic features (e.g., Country, City, State)
  - Supplier-related features (e.g., Distributor ID, Distributor Group, Manufacturer ID)
  - Hierarchical category features (Category Level 1–4)
  - A numeric feature Spend Random Factor which serves as a proxy for transaction magnitude


In [17]:
first_chunk.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 23 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Year Name                   100000 non-null  int64  
 1   Year Month                  100000 non-null  int64  
 2   Business Entity Type        100000 non-null  object 
 3   Country                     100000 non-null  object 
 4   Customer Market Segment Id  100000 non-null  object 
 5   Client ID                   99999 non-null   object 
 6   Customer Id                 100000 non-null  object 
 7   Customer Brand Id           79016 non-null   object 
 8   Customer Brand Parent Id    27365 non-null   object 
 9   City                        100000 non-null  object 
 10  State                       100000 non-null  object 
 11  Zip                         100000 non-null  object 
 12  Number of Rooms             100000 non-null  int64  
 13  Ecommerce Statu

- The .info() output reveals:
  - Several columns contain missing values, especially:
    - Customer Brand Parent Id
    - Ecommerce Status
    - Manufacturer ID
  - Most categorical variables are stored as object type
  - Numeric columns include Year, Year Month, Number of Rooms, and Spend Random Factor

This helps us understand both the structure and limitations of the dataset.

### Next Steps

Based on this, the next steps include:

1. Selecting a subset of relevant columns to reduce memory usage
2. Defining a clear analytical objective (e.g., supplier analysis, category spend, or geographic trends)
3. Processing the dataset in chunks to compute aggregated metrics instead of storing raw data

This ensures efficiency when working with a large dataset

## Missing Values and Early Profiling

After inspecting the structure of the first chunk, next we can assess data completeness and identify which columns are reliable enough for analysis. Since the full dataset contains approximately 44 million rows, we use the first chunk as an initial approximation of column quality.

This step helps determine:

- which columns contain substantial missingness
- which columns are suitable for aggregation
- which fields may require exclusion so reduce the number of features


In [18]:
missing_summary = first_chunk.isnull().sum().sort_values(ascending=False)
missing_summary

,0
Ecommerce Status,87958
Customer Brand Parent Id,72635
Manufacturer ID,34121
Customer Brand Id,20984
Category Level 4,10630
Category Level 3,7413
Distributor Group,4496
Distributor ID,3512
Category Level 2,1739
Client ID,1


## Missing Value Analysis

The dataset was examined for missing values using the first chunk as a representative sample. This step is important to identify unreliable features that could negatively impact our analysis.

The analysis reveals that certain columns contain a very high proportion of missing values:

- Ecommerce Status (~88% missing)
- Customer Brand Parent Id (~72% missing)

These columns are not suitable for analysis and will be excluded from further processing.

Other columns such as Manufacturer ID and Customer Brand Id contain moderate levels of missing data. These will be treated cautiously and may be excluded from certain analyses or handled using appropriate imputation techniques.

Importantly, several key columns have zero to almost no missing values, including:

- Year Month
- Geographic attributes (Country, State, City)
- Business Entity Type
- Category Level 1
- Category Level 2
- Spend Random Factor

These columns form a reliable foundation for both aggregation and machine learning tasks.

Additional features that were removed eve though they had no missing values:

- Client ID: This is a unique identifier for each client and does not contribute meaningful information for aggregation or clustering. Its high cardinality would introduce noise into the analysis.
- Category ID: This appears to be an internal identifier corresponding to category hierarchies. Since Category Level 1 and Category Level 2 already provide meaningful, human-readable groupings, this column is redundant.

Removing such identifier fields is important because:
- they do not provide analytical value
- they increase dimensionality unnecessarily
- they negatively impact machine learning models

### Next Steps

Based on this analysis, the next step is to:
- Select a subset of high-quality columns
- Reduce dimensionality for efficient processing
- Begin chunk-based aggregation for scalable analysis

In [19]:
clean_cols = [
    "Year Month",
    "Business Entity Type",
    "Country",
    "State",
    "City",
    "Customer Market Segment Id",
    "Number of Rooms",
    "Distributor Group",
    "Category Level 1",
    "Category Level 2",
    "Spend Random Factor"
]

In [20]:
#Chunking the dataset using only the reduced clean colums

chunk_test = next(pd.read_csv(fs.open(file_path, "rb"), usecols=clean_cols, chunksize=100000))
chunk_test.head()

,Year Month,Business Entity Type,Country,Customer Market Segment Id,City,State,Number of Rooms,Distributor Group,Category Level 1,Category Level 2,Spend Random Factor
0,202510,GPO,US,MS-100025,PINEHURST,NC,439,TURF MANAGEMENT,GOLF EQUIPMENT AND SUPPLIES,SPECIALIZED VEHICLES,1.237354e+06
1,202512,GPO,US,MS-100026,LAHAINA,HI,505,ENGINEERING CLEANING SERVICES,CHEMICALS AND CLEANING,CLEANING AND JANITORIAL SERVICES,6.002968e+05
2,202507,MANAGED SERVICES,US,MS-100007,GUILFORD,CT,0,ENGINEERING OTHER,MAINTENANCE AND ENGINEERING,M&E SERVICES,5.862804e+05
3,202505,GPO,US,MS-100026,LAHAINA,HI,505,ENGINEERING CLEANING SERVICES,CHEMICALS AND CLEANING,CLEANING AND JANITORIAL SERVICES,5.756939e+05
4,202510,GPO,US,MS-100026,LAHAINA,HI,505,ENGINEERING CLEANING SERVICES,CHEMICALS AND CLEANING,CLEANING AND JANITORIAL SERVICES,5.753405e+05


In [21]:
from collections import defaultdict

dist_spend = defaultdict(float)
dist_count = defaultdict(int)

chunk_size = 100000

with fs.open(file_path, "rb") as f:
    for i, chunk in enumerate(pd.read_csv(f, usecols=clean_cols, chunksize=chunk_size)):

        chunk.columns = [c.strip() for c in chunk.columns]

        chunk["Distributor Group"] = chunk["Distributor Group"].fillna("Unknown")

        grouped = chunk.groupby("Distributor Group")["Spend Random Factor"].agg(["sum", "count"])

        for idx, row in grouped.iterrows():
            dist_spend[idx] += row["sum"]
            dist_count[idx] += int(row["count"])

        if (i + 1) % 25 == 0:
            print(f"Processed {i+1} chunks")

Processed 25 chunks
Processed 50 chunks
Processed 75 chunks
Processed 100 chunks
Processed 125 chunks
Processed 150 chunks
Processed 175 chunks
Processed 200 chunks
Processed 225 chunks
Processed 250 chunks
Processed 275 chunks
Processed 300 chunks
Processed 325 chunks
Processed 350 chunks
Processed 375 chunks
Processed 400 chunks
Processed 425 chunks


In [23]:
dist_df = pd.DataFrame({
    "Distributor Group": list(dist_spend.keys()),
    "Total Spend": list(dist_spend.values()),
    "Transaction Count": [dist_count[k] for k in dist_spend.keys()]
})

dist_df = dist_df.sort_values("Total Spend", ascending=False)

dist_df.head(10)

,Distributor Group,Total Spend,Transaction Count
12,MASTER DISTRIBUTOR,3.277687e+09,28398552
8,ENGINEERING MRO,3.748054e+08,3321250
2,DIRECT BEVERAGE,3.096215e+08,1229770
20,REGIONAL PRODUCE,3.094375e+08,3035950
22,RETAIL,2.028612e+08,1698725
26,ROOM OPERATIONS - OTHER,1.950658e+08,938291
30,SPECIALTY FOODS,1.935986e+08,1064243
19,REGIONAL MEAT/POULTRY,1.891870e+08,491074
9,ENGINEERING OTHER,1.155428e+08,199098
32,Unknown,1.065176e+08,62320


## Distributor Group Spend Analysis (Chunk-Based Aggregation)

To analyze spending patterns across distributor groups, a scalable chunk-based aggregation approach was implemented.

### Methodology

Since the dataset contains approximately 44 million rows, it is not feasible to load the entire dataset into memory. Instead, the data was processed incrementally in chunks of 100,000 rows.

For each chunk:
- The Distributor Group column was cleaned by replacing missing values with "Unknown"
- Data was grouped by Distributor Group
- Two key metrics were computed:
  - Total spend proxy (Spend Random Factor)
  - Transaction count (number of records)

These intermediate results were accumulated across all chunks using dictionary-based aggregation. This ensures that the final output represents the entire dataset rather than just a subset.

### Key Results

The resulting table shows the top distributor groups ranked by total spend proxy.

Notable observations include:

- MASTER DISTRIBUTOR dominates significantly, with a total spend proxy exceeding 3.27 billion and over 28 million transactions
- Other major contributors include:
  - ENGINEERING MRO
  - DIRECT BEVERAGE
  - REGIONAL PRODUCE
  - RETAIL
- The distribution indicates a strong concentration of activity among a small number of distributor groups
- The presence of an "Unknown" category suggests some missing or unclassified distributor data

### Insights

- A small number of distributor groups account for a large proportion of total spend, indicating possible supplier concentration
- High transaction counts for certain groups suggest frequent procurement activity
- Differences between total spend and transaction count may indicate variation in average transaction size across groups

### Next Steps

To build a more complete understanding of the dataset, the next step is to perform similar chunk-based aggregations for:

- Category Level 1 (to identify key product/service categories)
- Geographic features such as State and City (to analyze regional distribution)
- Time (Year Month) to understand trends over time
